# Spark et Databricks (Partie 2) : créer, lire et écrire des DataFrames

📖 **Le mini-cours complet** : [Spark et Databricks : créer, lire et écrire des DataFrames PySpark (Partie 2)](https://www.monshotdata.com/p/spark-databricks-dataframes-pyspark-partie-2)

🧭 **Le cours** : [Bonnes pratiques d'ingénierie › 1.4 Passage à l'échelle et calcul distribué](https://www.monshotdata.com/mini-cours/bonnes-pratiques-ingenierie#h2sc14)

---

Le DataFrame est la structure que tu manipules au quotidien avec Spark : une table distribuée, immuable et paresseuse, avec un schéma où chaque colonne a un type précis. Ici tu crées des DataFrames de toutes les façons possibles, tu lis et écris des fichiers, puis tu les affiches.

**À la fin de ce notebook, tu sauras :**

- créer un DataFrame à partir d'une liste, d'objets `Row` ou d'un schéma vide, et déclarer ce schéma (`StructType` ou syntaxe DDL) ;
- lire du CSV et du Parquet, et éviter le piège de `load()` sur Databricks ;
- écrire un DataFrame et comprendre pourquoi Spark produit un dossier, pas un fichier ;
- choisir entre `show()`, `head()`, `limit()` et `tail()`, et mesurer un DataFrame.

### Avant de lancer

Ce notebook tourne sur **Databricks**, pas sur Colab. Un compte **Databricks Free Edition** (gratuit) suffit.

- **Ouvrir le notebook** : dans Databricks, Workspace → Import, puis dépose ce fichier `.ipynb` (ou colle l'URL du fichier sur GitHub).
- **Le lancer** : attache-le au calcul **serverless**, puis exécute les cellules de haut en bas (ou « Run all »).
- **Les données** : la première cellule recrée le Volume `workspace.default.cours_spark` et le fichier `employes.csv` de la Partie 1 : tu peux lancer ce notebook sans avoir fait la Partie 1. Les cellules d'écriture ajoutent des dossiers dans ce Volume.
- **Les erreurs voulues** : quelques cellules provoquent l'erreur expliquée dans le mini-cours. Elle est capturée par un `try / except` et affichée, pour que l'exécution aille au bout.

🧪 *Testé le 21 novembre 2026 : exécution complète de haut en bas sur Databricks Free Edition (calcul serverless, Spark 4.2, Python 3.12). Les sorties affichées viennent de cette exécution ; les identifiants techniques (adresse de la session, noms de fichiers `part-…`, `uid` des modèles) changent à chaque lancement.*

## 1. Préparation

Ce notebook s'exécute seul : cette cellule recrée le Volume et le fichier `employes.csv` de la Partie 1 (même code).

In [1]:
import numpy as np
import pandas as pd

# Crée un Volume : un espace de stockage de fichiers dans Unity Catalog
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.cours_spark")
chemin_volume = "/Volumes/workspace/default/cours_spark"

# Dataset fictif de 1 000 employés (graine fixe : résultats reproductibles)
rng = np.random.default_rng(42)
n = 1000
prenoms = ["Léa", "Hugo", "Emma", "Louis", "Chloé",
           "Lucas", "Inès", "Nathan", "Jade", "Adam"]
noms = ["Martin", "Bernard", "Dubois", "Thomas", "Robert",
        "Richard", "Petit", "Durand", "Leroy", "Moreau"]
villes = ["Paris", "Lyon", "Marseille", "Toulouse", "Bordeaux", "Lille"]
postes = ["Data Analyst", "Data Scientist", "Data Engineer",
          "ML Engineer", "Manager"]

employes = pd.DataFrame({
    "id_employe": np.arange(1001, 1001 + n),
    "prenom": rng.choice(prenoms, n),
    "nom": rng.choice(noms, n),
    "age": rng.integers(22, 61, n),
    "ville": rng.choice(villes, n),
    "poste": rng.choice(postes, n),
    "salaire": rng.integers(32, 95, n) * 1000,
})

# Écrit le fichier CSV dans le Volume
employes.to_csv(f"{chemin_volume}/employes.csv", index=False)
print(employes.shape)

# Résultat :
# (1000, 7)

(1000, 7)


## 2. La SparkSession, ta porte d'entrée

Récupère la session Spark : dans Databricks, l'objet `spark` existe déjà.

In [2]:
from pyspark.sql import SparkSession

# Récupère la session existante (ou en crée une hors de Databricks)
spark = SparkSession.builder.appName("SparkDF").getOrCreate()
print(type(spark))

# Résultat :
# <class 'pyspark.sql.connect.session.SparkSession'>

<class 'pyspark.sql.connect.session.SparkSession'>


## 3. Les objets Row

Une ligne d'un DataFrame est un objet `Row`, créé avec des arguments nommés.

In [3]:
from pyspark.sql import Row

# Une ligne = un objet Row, créé avec des arguments nommés
ligne_a = Row(nom="Pierre", age=22, ville="Lyon")

print(ligne_a)          # la ligne complète
print(ligne_a.nom)      # accès par nom de colonne
print(ligne_a["age"])   # accès par clé
print(ligne_a[2])       # accès par position

# Résultat :
# Row(nom='Pierre', age=22, ville='Lyon')
# Pierre
# 22
# Lyon

Row(nom='Pierre', age=22, ville='Lyon')
Pierre
22
Lyon


In [4]:
# Valeur manquante : déclare-la explicitement avec None
ligne_b = Row(nom="Marc", age=25, ville=None)
print(ligne_b)

# Un modèle de ligne réutilisable : plus besoin de répéter les noms
Personne = Row("nom", "age", "ville")

lignes = [
    Personne("Pierre", 22, "Lyon"),
    Personne("Marc", 25, "Londres"),
    Personne("Julie", 26, "Dubaï"),
    Personne("Katia", 23, "Sydney"),
]
print(lignes[0])

# Résultat :
# Row(nom='Marc', age=25, ville=None)
# Row(nom='Pierre', age=22, ville='Lyon')

Row(nom='Marc', age=25, ville=None)
Row(nom='Pierre', age=22, ville='Lyon')


## 4. Créer un DataFrame en mémoire

Trois méthodes en mémoire : liste, objets `Row`, DataFrame vide, avec ou sans schéma explicite. Les erreurs montrées dans le mini-cours sont capturées par un `try/except`, pour que « Run all » aille au bout.

In [5]:
donnees = [("Pierre", 22, "Lyon"), ("Marc", 25, "Londres"),
           ("Julie", 26, "Dubaï"), ("Katia", 23, "Sydney")]

df = spark.createDataFrame(donnees)
print(type(df))
print(df)

# Résultat :
# <class 'pyspark.sql.connect.dataframe.DataFrame'>
# DataFrame[_1: string, _2: bigint, _3: string]

<class 'pyspark.sql.connect.dataframe.DataFrame'>
DataFrame[_1: string, _2: bigint, _3: string]


In [6]:
# show() est une action : elle déclenche le calcul
df.show()

# toDF() renvoie un nouveau DataFrame avec les noms de colonnes voulus
df = spark.createDataFrame(donnees).toDF("Nom", "Age", "Ville")
df.show()

# Résultat :
# +------+---+-------+
# |    _1| _2|     _3|
# +------+---+-------+
# |Pierre| 22|   Lyon|
# |  Marc| 25|Londres|
# | Julie| 26|  Dubaï|
# | Katia| 23| Sydney|
# +------+---+-------+
#
# +------+---+-------+
# |   Nom|Age|  Ville|
# +------+---+-------+
# |Pierre| 22|   Lyon|
# |  Marc| 25|Londres|
# | Julie| 26|  Dubaï|
# | Katia| 23| Sydney|
# +------+---+-------+

+------+---+-------+
|    _1| _2|     _3|
+------+---+-------+
|Pierre| 22|   Lyon|
|  Marc| 25|Londres|
| Julie| 26|  Dubaï|
| Katia| 23| Sydney|
+------+---+-------+

+------+---+-------+
|   Nom|Age|  Ville|
+------+---+-------+
|Pierre| 22|   Lyon|
|  Marc| 25|Londres|
| Julie| 26|  Dubaï|
| Katia| 23| Sydney|
+------+---+-------+



In [7]:
# Quelques valeurs manquantes : Spark déduit quand même le type
donnees_nulles = [("Pierre", 22, None), ("Marc", 25, "Londres"),
                  ("Katia", 23, None)]
spark.createDataFrame(donnees_nulles).toDF("Nom", "Age", "Ville").show()

# +------+---+-------+
# |   Nom|Age|  Ville|
# +------+---+-------+
# |Pierre| 22|   NULL|
# |  Marc| 25|Londres|
# | Katia| 23|   NULL|
# +------+---+-------+

# Une colonne entièrement vide : Spark ne peut pas deviner son type
toutes_nulles = [("Pierre", 22, None), ("Marc", 25, None), ("Katia", 23, None)]
try:
    spark.createDataFrame(toutes_nulles)
except Exception as erreur:
    print(type(erreur).__name__, ":", str(erreur)[:150])

# PySparkValueError: [CANNOT_DETERMINE_TYPE] Some of types cannot be determined
# after inferring.

+------+---+-------+
|   Nom|Age|  Ville|
+------+---+-------+
|Pierre| 22|   NULL|
|  Marc| 25|Londres|
| Katia| 23|   NULL|
+------+---+-------+

PySparkValueError : [CANNOT_DETERMINE_TYPE] Some of types cannot be determined after inferring.


In [8]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

schema = StructType([
    StructField("Nom", StringType(), True),    # True : valeurs nulles autorisées
    StructField("Age", IntegerType(), True),
    StructField("Ville", StringType(), True),
])

df_schema = spark.createDataFrame(toutes_nulles, schema=schema)
df_schema.printSchema()
df_schema.show()

# Résultat :
# root
#  |-- Nom: string (nullable = true)
#  |-- Age: integer (nullable = true)
#  |-- Ville: string (nullable = true)
#
# +------+---+-----+
# |   Nom|Age|Ville|
# +------+---+-----+
# |Pierre| 22| NULL|
# |  Marc| 25| NULL|
# | Katia| 23| NULL|
# +------+---+-----+

root
 |-- Nom: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Ville: string (nullable = true)

+------+---+-----+
|   Nom|Age|Ville|
+------+---+-----+
|Pierre| 22| NULL|
|  Marc| 25| NULL|
| Katia| 23| NULL|
+------+---+-----+



In [9]:
# Méthode 2 : à partir d'une liste d'objets Row
df_lignes = spark.createDataFrame(lignes)
df_lignes.show()

# +------+---+-------+
# |   nom|age|  ville|
# +------+---+-------+
# |Pierre| 22|   Lyon|
# |  Marc| 25|Londres|
# | Julie| 26|  Dubaï|
# | Katia| 23| Sydney|
# +------+---+-------+

# Une ligne sans « ville » : le DataFrame ne peut pas être construit
lignes_incompletes = [Row(nom="Pierre", age=22, ville="Lyon"),
                      Row(nom="Marc", age=25)]
try:
    spark.createDataFrame(lignes_incompletes).show()
except Exception as erreur:
    print(type(erreur).__name__, ":", str(erreur)[:150])

# PySparkValueError: [AXIS_LENGTH_MISMATCH] Length mismatch:
# Expected axis has 3 elements, new values have 2 elements.

# Avec ville=None, tout fonctionne
spark.createDataFrame([Row(nom="Pierre", age=22, ville="Lyon"),
                       Row(nom="Marc", age=25, ville=None)]).show()

# +------+---+-----+
# |   nom|age|ville|
# +------+---+-----+
# |Pierre| 22| Lyon|
# |  Marc| 25| NULL|
# +------+---+-----+

+------+---+-------+
|   nom|age|  ville|
+------+---+-------+
|Pierre| 22|   Lyon|
|  Marc| 25|Londres|
| Julie| 26|  Dubaï|
| Katia| 23| Sydney|
+------+---+-------+

PySparkValueError : [AXIS_LENGTH_MISMATCH] Length mismatch: Expected axis has 3 elements, new values have 2 elements.
+------+---+-----+
|   nom|age|ville|
+------+---+-----+
|Pierre| 22| Lyon|
|  Marc| 25| NULL|
+------+---+-----+



In [10]:
# Méthode 3 : un DataFrame vide, schéma en DDL (syntaxe SQL)
df_vide = spark.createDataFrame([], "col1 INT, col2 STRING")
df_vide.printSchema()
print(df_vide.count())

# La même syntaxe DDL pour nos données
spark.createDataFrame(donnees, "Nom STRING, Age INT, Ville STRING").printSchema()

# Résultat :
# root
#  |-- col1: integer (nullable = true)
#  |-- col2: string (nullable = true)
#
# 0
# root
#  |-- Nom: string (nullable = true)
#  |-- Age: integer (nullable = true)
#  |-- Ville: string (nullable = true)

root
 |-- col1: integer (nullable = true)
 |-- col2: string (nullable = true)

0
root
 |-- Nom: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Ville: string (nullable = true)


## 5. Lire et écrire des fichiers

Lis du CSV et du Parquet depuis le Volume, puis écris un DataFrame en CSV.

In [11]:
chemin_volume = "/Volumes/workspace/default/cours_spark"

# Forme générique : format + option + load
df_csv = (
    spark.read.format("csv")
    .option("header", "true")      # "true" ou True : les deux marchent
    .option("inferSchema", True)
    .load(f"{chemin_volume}/employes.csv")
)
df_csv.printSchema()

# Résultat :
# root
#  |-- id_employe: integer (nullable = true)
#  |-- prenom: string (nullable = true)
#  |-- nom: string (nullable = true)
#  |-- age: integer (nullable = true)
#  |-- ville: string (nullable = true)
#  |-- poste: string (nullable = true)
#  |-- salaire: integer (nullable = true)

root
 |-- id_employe: integer (nullable = true)
 |-- prenom: string (nullable = true)
 |-- nom: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- ville: string (nullable = true)
 |-- poste: string (nullable = true)
 |-- salaire: integer (nullable = true)


In [12]:
# Sans options : colonnes _c0, _c1... et tout en chaînes de caractères
df_brut = spark.read.format("csv").load(f"{chemin_volume}/employes.csv")
print(df_brut.dtypes[:3])
# [('_c0', 'string'), ('_c1', 'string'), ('_c2', 'string')]

# Schéma connu à l'avance (syntaxe DDL) : pas d'inférence, lecture plus rapide
schema_employes = ("id_employe INT, prenom STRING, nom STRING, age INT, "
                   "ville STRING, poste STRING, salaire INT")
df_csv = spark.read.csv(f"{chemin_volume}/employes.csv",
                        header=True, schema=schema_employes)
df_csv.printSchema()   # même schéma qu'avec inferSchema, sans l'inférence

[('_c0', 'string'), ('_c1', 'string'), ('_c2', 'string')]
root
 |-- id_employe: integer (nullable = true)
 |-- prenom: string (nullable = true)
 |-- nom: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- ville: string (nullable = true)
 |-- poste: string (nullable = true)
 |-- salaire: integer (nullable = true)


In [13]:
# Écris le DataFrame en Parquet : le schéma est stocké avec les données
df_csv.write.mode("overwrite").parquet(f"{chemin_volume}/employes_parquet")

# Relis-le : aucune option nécessaire
df_parquet = spark.read.parquet(f"{chemin_volume}/employes_parquet")
print(df_parquet.count(), df_parquet.columns)
# 1000 ['id_employe', 'prenom', 'nom', 'age', 'ville', 'poste', 'salaire']

# Sur Databricks, load() sans format lit du Delta par défaut :
try:
    spark.read.load(f"{chemin_volume}/employes_parquet").count()
except Exception as erreur:
    print(type(erreur).__name__, ":", str(erreur)[:150])
# AnalysisException: [DELTA_MISSING_TRANSACTION_LOG] Incompatible format detected.

1000 ['id_employe', 'prenom', 'nom', 'age', 'ville', 'poste', 'salaire']


AnalysisException : [DELTA_MISSING_TRANSACTION_LOG] Incompatible format detected.

You are trying to read from `/Volumes/workspace/default/cours_spark/employes_parquet` u


In [14]:
# Lire tout un dossier, en ne gardant que les fichiers .parquet
df_dossier = (
    spark.read.option("pathGlobFilter", "*.parquet")
    .parquet(f"{chemin_volume}/employes_parquet")
)
print(df_dossier.count())

# Résultat :
# 1000

1000


In [15]:
# Écriture : mode overwrite, append, ignore ou error (par défaut)
(
    df.write.mode("overwrite")
    .format("csv")
    .option("header", True)
    .save(f"{chemin_volume}/personnes_csv")
)
print(sorted(f.name for f in dbutils.fs.ls(f"{chemin_volume}/personnes_csv")))

# Résultat (noms raccourcis) :
# ['_SUCCESS', '_committed_3620...', '_started_3620...',
#  'part-00000-tid-3620...-c000.csv', 'part-00001-tid-3620...-c000.csv',
#  'part-00002-tid-3620...-c000.csv', 'part-00003-tid-3620...-c000.csv']

['_committed_3620846374814282767', '_committed_42890949831849303', '_committed_5205433623708395344', '_committed_vacuum1016777584115507818', '_started_5205433623708395344', 'part-00000-tid-5205433623708395344-1f10d752-6f20-4c85-be2a-3e2022d9b41b-144-1-c000.csv', 'part-00001-tid-5205433623708395344-1f10d752-6f20-4c85-be2a-3e2022d9b41b-145-1-c000.csv', 'part-00002-tid-5205433623708395344-1f10d752-6f20-4c85-be2a-3e2022d9b41b-146-1-c000.csv', 'part-00003-tid-5205433623708395344-1f10d752-6f20-4c85-be2a-3e2022d9b41b-147-1-c000.csv']


## 6. Afficher, extraire et mesurer un DataFrame

`show()`, `head()`, `limit()`, `tail()` et les dimensions d'un DataFrame.

In [16]:
df.show(2)                  # les 2 premières lignes
df.show(2, vertical=True)   # affichage vertical, pratique pour les tables larges

resultat = df.show(1)
print(resultat)             # show() affiche, mais ne renvoie rien

# Résultat :
# +------+---+-------+
# |   Nom|Age|  Ville|
# +------+---+-------+
# |Pierre| 22|   Lyon|
# |  Marc| 25|Londres|
# +------+---+-------+
# only showing top 2 rows
#
# -RECORD 0--------
#  Nom   | Pierre
#  Age   | 22
#  Ville | Lyon
# -RECORD 1--------
#  Nom   | Marc
#  Age   | 25
#  Ville | Londres
# only showing top 2 rows
#
# (show(1) affiche la première ligne, puis :)
# None

+------+---+-------+
|   Nom|Age|  Ville|
+------+---+-------+
|Pierre| 22|   Lyon|
|  Marc| 25|Londres|
+------+---+-------+
only showing top 2 rows
-RECORD 0--------
 Nom   | Pierre  
 Age   | 22      
 Ville | Lyon    
-RECORD 1--------
 Nom   | Marc    
 Age   | 25      
 Ville | Londres 
only showing top 2 rows
+------+---+-----+
|   Nom|Age|Ville|
+------+---+-----+
|Pierre| 22| Lyon|
+------+---+-----+
only showing top 1 row
None


In [17]:
print(df.head(2))        # liste d'objets Row
df.limit(2).show()       # un DataFrame
print(df.tail(2))        # liste d'objets Row (dernières lignes)

# Transformer les dernières lignes en DataFrame
df_fin = spark.createDataFrame(df.tail(2), schema=df.schema)
df_fin.show()

# Résultat :
# [Row(Nom='Pierre', Age=22, Ville='Lyon'),
#  Row(Nom='Marc', Age=25, Ville='Londres')]
# +------+---+-------+
# |   Nom|Age|  Ville|
# +------+---+-------+
# |Pierre| 22|   Lyon|
# |  Marc| 25|Londres|
# +------+---+-------+
#
# [Row(Nom='Julie', Age=26, Ville='Dubaï'),
#  Row(Nom='Katia', Age=23, Ville='Sydney')]
# +-----+---+------+
# |  Nom|Age| Ville|
# +-----+---+------+
# |Julie| 26| Dubaï|
# |Katia| 23|Sydney|
# +-----+---+------+

[Row(Nom='Pierre', Age=22, Ville='Lyon'), Row(Nom='Marc', Age=25, Ville='Londres')]
+------+---+-------+
|   Nom|Age|  Ville|
+------+---+-------+
|Pierre| 22|   Lyon|
|  Marc| 25|Londres|
+------+---+-------+

[Row(Nom='Julie', Age=26, Ville='Dubaï'), Row(Nom='Katia', Age=23, Ville='Sydney')]
+-----+---+------+
|  Nom|Age| Ville|
+-----+---+------+
|Julie| 26| Dubaï|
|Katia| 23|Sydney|
+-----+---+------+



In [18]:
nb_lignes = df.count()
nb_colonnes = len(df.columns)
print(f"Dimensions : ({nb_lignes}, {nb_colonnes})")
print(df.isEmpty())
print(df.columns)

# Résultat :
# Dimensions : (4, 3)
# False
# ['Nom', 'Age', 'Ville']

Dimensions : (4, 3)
False
['Nom', 'Age', 'Ville']


## 7. Ce qu'il faut retenir

- Un DataFrame Spark est immuable, distribué et paresseux : chaque transformation renvoie un nouveau DataFrame, et rien ne s'exécute avant une action.
- La SparkSession est ton point d'entrée ; dans Databricks, l'objet `spark` existe déjà.
- Déclare ton schéma (StructType ou DDL) : tu évites l'inférence sur les gros volumes et l'erreur des colonnes entièrement vides.
- Précise toujours le format de lecture : sur Databricks, `load()` lit du Delta par défaut.
- `show()` affiche sans rien renvoyer ; `head()` et `tail()` renvoient des objets Row, `limit()` un DataFrame.

---

## Pour aller plus loin

- [Spark et Databricks : du big data à ton premier notebook PySpark (Partie 1)](https://www.monshotdata.com/p/spark-databricks-calcul-distribue-partie-1) — la partie précédente : Big data, calcul distribué, RDD et premier notebook PySpark
- [Spark et Databricks : maîtriser colonnes, lignes et tris en PySpark (Partie 3)](https://www.monshotdata.com/p/spark-databricks-colonnes-lignes-tris-partie-3) — la partie suivante : sélectionner, transformer, filtrer et trier